# 09 — Export, Inférence et Déploiement — BiLSTM

**Prérequis** : `08_optimisation.ipynb` exécuté → `bilstm_optimised.pt` disponible.

**Données** : `../data/client_data.pkl`, `../data/vocab.pkl`, `../data/label_maps.json`

**Structure (pattern TP10)** :
```
Partie 0 : Rechargement du modèle BiLSTM optimisé
Partie 1 : TorchScript (trace)
Partie 2 : Export ONNX
Partie 3 : Quantization dynamique
Partie 4 : Benchmark — 3 formats
Partie 5 : Fonction d'inférence sur texte brut
Bonus    : Mini API FastAPI /predict
```

## Partie 0 — Rechargement du modèle (pattern TP10)

In [ ]:
import os, json, time, pickle, re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

DATA_DIR   = '../data/'
MODEL_DIR  = '../models/'
EXPORT_DIR = '../exports/'
os.makedirs(EXPORT_DIR, exist_ok=True)

MAX_LEN    = 64
BATCH_SIZE = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'PyTorch : {torch.__version__}')

try:
    import onnx
    import onnxruntime as ort
    print('ONNX Runtime disponible ✓')
except ImportError:
    print('ONNX non installé — pip install onnx onnxruntime')

In [ ]:
# Chargement des artefacts du notebook 01
with open(DATA_DIR + 'client_data.pkl', 'rb') as f:
    data = pickle.load(f)
with open(DATA_DIR + 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
N_CLASSES   = len(CLIENT_LABELS)
label_names = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX     = vocab['<PAD>']

print(f'Vocab : {len(vocab):,} tokens')
print(f'N_CLASSES = {N_CLASSES} — {label_names}')

In [ ]:
# Dataset + DataLoader test (pattern TP5)
class IntentDataset(Dataset):
    def __init__(self, data):
        self.texts  = [d[0] for d in data]
        self.labels = [d[1] for d in data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded  = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

test_loader = DataLoader(
    IntentDataset(data['test']),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
print(f'Test : {len(data["test"]):,} exemples')

In [ ]:
# Architecture BiLSTM (identique aux notebooks 03 et 08)
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim,
                 num_layers=2, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim,
                                 num_layers=num_layers, batch_first=True,
                                 bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded       = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        last_hidden    = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(last_hidden))


# Rechargement depuis bilstm_optimised.pt
# map_location='cpu' : nécessaire pour déploiement sans GPU et pour TorchScript/ONNX
model = BiLSTM(vocab_size=len(vocab), embed_dim=128, hidden_dim=256,
               output_dim=N_CLASSES, num_layers=2, dropout=0.3)
model.load_state_dict(torch.load(MODEL_DIR + 'bilstm_optimised.pt', map_location='cpu'))
model.eval()
print(f'Paramètres : {sum(p.numel() for p in model.parameters()):,}')

# Accuracy de référence
def evaluate(model_fn, loader):
    preds, targets = [], []
    with torch.no_grad():
        for texts, labels in loader:
            logits = model_fn(texts)
            preds.extend(torch.argmax(logits, dim=1).tolist())
            targets.extend(labels.tolist())
    return accuracy_score(targets, preds)

acc_base = evaluate(model, test_loader)
print(f'Accuracy baseline (PyTorch) : {acc_base:.4f}')

## Partie 1 — TorchScript (pattern TP10)

Le BiLSTM est une architecture **statique** (pas de branches conditionnelles dans `forward`)
→ `torch.jit.trace` suffit.

Le fichier `.pt` TorchScript peut être chargé et exécuté **en C++ sans Python installé**.

In [ ]:
# Entrée dummy : batch_size=1, seq_len=MAX_LEN
dummy = torch.ones(1, MAX_LEN, dtype=torch.long)

model.eval()
traced    = torch.jit.trace(model, dummy)
path_ts   = os.path.join(EXPORT_DIR, 'bilstm_traced.pt')
traced.save(path_ts)
size_ts   = os.path.getsize(path_ts) / (1024**2)
print(f'TorchScript sauvegardé — taille : {size_ts:.1f} Mo')

# Rechargement et vérification : accuracy doit être identique
model_ts  = torch.jit.load(path_ts)
model_ts.eval()
acc_ts    = evaluate(model_ts, test_loader)
print(f'Accuracy TorchScript : {acc_ts:.4f} (base={acc_base:.4f})')

## Partie 2 — Export ONNX (pattern TP10)

ONNX permet d'exécuter le BiLSTM **sans PyTorch** : C++, Java, mobile, Azure, OpenVINO.

`dynamic_axes` est essentiel : sans ça le modèle n'accepterait qu'un `batch_size`
et `seq_len` fixés à l'export.

In [ ]:
path_onnx = os.path.join(EXPORT_DIR, 'bilstm.onnx')

torch.onnx.export(
    model,
    dummy,
    path_onnx,
    export_params=True,
    opset_version=14,
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'seq_len'},
        'logits':    {0: 'batch_size'}
    }
)
size_onnx = os.path.getsize(path_onnx) / (1024**2)
print(f'ONNX exporté — taille : {size_onnx:.1f} Mo')

try:
    import onnx
    onnx.checker.check_model(onnx.load(path_onnx))
    print('ONNX valide ✓')
except Exception as e:
    print(f'Validation : {e}')

In [ ]:
# Inférence ONNX Runtime — indépendant de PyTorch
try:
    sess      = ort.InferenceSession(path_onnx)
    preds, targets = [], []
    for texts, labels in test_loader:
        output = sess.run(['logits'], {'input_ids': texts.numpy()})[0]
        preds.extend(output.argmax(axis=1).tolist())
        targets.extend(labels.tolist())
    acc_onnx = accuracy_score(targets, preds)
    print(f'Accuracy ONNX Runtime : {acc_onnx:.4f}')
except Exception as e:
    print(f'ONNX Runtime : {e}')
    acc_onnx = acc_base

## Partie 3 — Quantization dynamique (pattern TP10)

Réduit la précision des poids `float32` → `int8` : **~4× plus léger**, plus rapide sur CPU.

Le BiLSTM contient des couches `nn.Linear` (projections LSTM + tête FC)
→ toutes quantizables avec la quantization dynamique.

In [ ]:
model_q = torch.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)
model_q.eval()

path_q    = os.path.join(EXPORT_DIR, 'bilstm_quantized.pth')
torch.save(model_q, path_q)

size_base = os.path.getsize(MODEL_DIR + 'bilstm_optimised.pt') / (1024**2)
size_q    = os.path.getsize(path_q) / (1024**2)
print(f'Baseline : {size_base:.1f} Mo')
print(f'Quantizé : {size_q:.1f} Mo  (ratio : {size_base/size_q:.1f}×)')

acc_q = evaluate(model_q, test_loader)
print(f'Accuracy quantizé : {acc_q:.4f} (delta : {acc_q - acc_base:+.4f})')

## Partie 4 — Benchmark 3 formats (pattern TP10)

Taille, latence, accuracy sur une seule séquence (simulation production).

In [ ]:
def mesure_latence(fn, n=200):
    for _ in range(20): fn()       # warmup
    t0 = time.time()
    for _ in range(n): fn()
    return (time.time() - t0) / n * 1000   # ms

single = torch.ones(1, MAX_LEN, dtype=torch.long)

lat_base = mesure_latence(lambda: model(single))
lat_ts   = mesure_latence(lambda: model_ts(single))
lat_q    = mesure_latence(lambda: model_q(single))

results = [
    {'nom': 'PyTorch (base)',  'taille_mb': size_base, 'latence_ms': lat_base, 'accuracy': acc_base},
    {'nom': 'TorchScript',     'taille_mb': size_ts,   'latence_ms': lat_ts,   'accuracy': acc_ts},
    {'nom': 'Quantizé (int8)', 'taille_mb': size_q,    'latence_ms': lat_q,    'accuracy': acc_q},
]

print(f'{"Version":<22} {"Taille (Mo)":>12} {"Latence (ms)":>14} {"Accuracy":>10}')
print('─' * 62)
for r in results:
    print(f'{r["nom"]:<22} {r["taille_mb"]:>12.1f} {r["latence_ms"]:>14.2f} {r["accuracy"]:>10.4f}')

In [ ]:
# Visualisation (pattern TP10)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
noms       = [r['nom']        for r in results]
tailles    = [r['taille_mb']  for r in results]
latences   = [r['latence_ms'] for r in results]
accuracies = [r['accuracy']   for r in results]
colors     = ['steelblue', 'seagreen', 'coral']

for ax, vals, xlabel, title in [
    (axes[0], tailles,    'Taille (Mo)',  'Taille du modèle'),
    (axes[1], latences,   'Latence (ms)', 'Latence (1 tweet, CPU)'),
    (axes[2], accuracies, 'Accuracy',     'Accuracy test set'),
]:
    bars = ax.barh(noms, vals, color=colors)
    ax.set_xlabel(xlabel); ax.set_title(title); ax.invert_yaxis()
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=9)

plt.suptitle('Benchmark BiLSTM — 3 formats', fontsize=13)
plt.tight_layout()
plt.savefig(EXPORT_DIR + 'benchmark_bilstm.png', dpi=100)
plt.show()

## Partie 5 — Fonction d'inférence sur texte brut (pattern TP10)

Pipeline **identique** à `01_clustering.ipynb` :
```
Texte brut
    ↓  clean_tweet()   ← même fonction que Bloc 5 du notebook 01
    ↓  tokenize()      ← vocab custom, MAX_LEN=64
    ↓  pad_sequence()
    ↓  BiLSTM forward
    ↓  softmax → top-k
→ [(label, confiance), ...]
```

> Cohérence critique : utiliser un preprocessing différent en production
> dégraderait les performances même si le modèle est parfait.

In [ ]:
# Même preprocessing que 01_clustering.ipynb Bloc 5 — critique pour la cohérence train/prod
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r"[^\w\s?!']", ' ', text)  # préserve ? ! ' comme à l'entraînement
    return re.sub(r'\s+', ' ', text).strip()


# Même tokenize que 01_clustering.ipynb Bloc 13
def tokenize(text, max_len=64):
    tokens = text.split()[:max_len]
    ids    = [vocab.get(w, vocab['<UNK>']) for w in tokens]
    return ids if ids else [vocab['<UNK>']]


def predict(text, model, top_k=3):
    cleaned   = clean_tweet(text)
    token_ids = tokenize(cleaned)
    tensor    = pad_sequence(
        [torch.tensor(token_ids, dtype=torch.long)],
        batch_first=True, padding_value=PAD_IDX
    )
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
    top_val, top_idx = torch.topk(probs, k=top_k)
    return [(label_names[i.item()], round(v.item(), 4))
            for i, v in zip(top_idx, top_val)]


examples = [
    '@AppleSupport my iPhone keeps crashing since the update',
    '@AmericanAir my flight to Chicago is delayed 3 hours',
    '@AmazonHelp my order still not delivered after 10 days',
    '@Uber_Support driver cancelled my ride at the last minute',
    '@McDonalds the food was cold and the order was wrong',
    '@BofA_Help my card was charged twice for the same purchase',
    '@Comcast my internet has been down for 2 days',
    'Merci beaucoup pour votre aide rapide !',
    'thanks sent',
    '@VerizonSupport I have no signal at all in my area',
]

print(f'{"Tweet":<55} {"Top-1 prédit":<20} {"Confiance":>10}')
print('-' * 90)
for tweet in examples:
    preds = predict(tweet, model)
    print(f'{tweet[:54]:<55} {preds[0][0]:<20} {preds[0][1]:>10.2%}')

In [ ]:
# Inspection top-3 sur un tweet ambigu
tweet = '@VirginTrains the wifi is not working and connection keeps dropping'
preds = predict(tweet, model, top_k=3)
print(f'Tweet : {tweet}\n')
for rank, (label, conf) in enumerate(preds, 1):
    bar = '█' * int(conf * 40)
    print(f'  {rank}. {label:<20} {conf:.2%}  {bar}')

In [ ]:
# Rapport des fichiers exportés
print('Fichiers exportés :')
for fname in sorted(os.listdir(EXPORT_DIR)):
    path = os.path.join(EXPORT_DIR, fname)
    size = os.path.getsize(path) / (1024**2)
    print(f'  {fname:<40} {size:>6.1f} Mo')

## Bonus — Mini API FastAPI (pattern TP10)

```
POST /predict  →  {"intention": "problem_report", "confiance": 0.99, "top3": [...]}
GET  /health   →  {"status": "ok", "modele": "BiLSTM"}
```

**Lancement** :
```bash
pip install fastapi uvicorn
uvicorn api_bilstm:app --reload --port 8000
```

**Test** :
```bash
curl -X POST http://localhost:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"text": "my flight got cancelled and no one answered"}'
```

In [ ]:
api_code = '''
"""
API FastAPI — Classification d'intentions tweets (BiLSTM)
Lancer : uvicorn api_bilstm:app --reload --port 8000
Docs   : http://localhost:8000/docs
"""

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import pickle, json, re

app = FastAPI(title="Intent Classifier — BiLSTM", version="1.0")

VOCAB_PATH  = "../data/vocab.pkl"
LABELS_PATH = "../data/label_maps.json"
MODEL_PATH  = "../models/bilstm_optimised.pt"
MAX_LEN     = 64

with open(VOCAB_PATH, "rb") as f:
    vocab = pickle.load(f)
with open(LABELS_PATH) as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps["client"].items()}
N_CLASSES     = len(CLIENT_LABELS)
LABEL_NAMES   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX       = vocab["<PAD>"]


class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, output_dim)
    def forward(self, x):
        _, (h, _) = self.lstm(self.embedding(x))
        return self.fc(self.dropout(torch.cat([h[-2], h[-1]], dim=1)))


model = BiLSTM(len(vocab), 128, 256, N_CLASSES, 2, 0.3)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()


def clean_tweet(text):
    """Preprocessing identique à 01_clustering.ipynb Bloc 5."""
    text = str(text).lower()
    text = re.sub(r"@\\w+", "", text)
    text = re.sub(r"http\\S+", "", text)
    text = re.sub(r"#\\w+", "", text)
    text = re.sub(r"\\d+", "", text)
    text = re.sub(r"[^\\w\\s?!\\']", " ", text)
    return re.sub(r"\\s+", " ", text).strip()

def tokenize(text):
    tokens = text.split()[:MAX_LEN]
    return [vocab.get(w, vocab["<UNK>"]) for w in tokens] or [vocab["<UNK>"]]


class PredictionInput(BaseModel):
    text: str

class TopPrediction(BaseModel):
    intention: str
    confiance: float

class PredictionResponse(BaseModel):
    intention: str
    confiance: float
    top3: List[TopPrediction]


@app.get("/health")
def health():
    return {"status": "ok", "modele": "BiLSTM", "classes": N_CLASSES}


@app.post("/predict", response_model=PredictionResponse)
def predict(payload: PredictionInput):
    token_ids = tokenize(clean_tweet(payload.text))
    tensor    = pad_sequence(
        [torch.tensor(token_ids, dtype=torch.long)],
        batch_first=True, padding_value=PAD_IDX
    )
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    top_val, top_idx = torch.topk(probs, k=3)
    top3 = [TopPrediction(intention=LABEL_NAMES[i], confiance=round(v.item(), 4))
            for i, v in zip(top_idx, top_val)]
    return PredictionResponse(
        intention=top3[0].intention,
        confiance=top3[0].confiance,
        top3=top3
    )
'''

with open('api_bilstm.py', 'w') as f:
    f.write(api_code)

print('api_bilstm.py généré')
print('Installation : pip install fastapi uvicorn')
print('Lancement    : uvicorn api_bilstm:app --reload --port 8000')
print('Docs         : http://localhost:8000/docs')